In [9]:
import json
import os
import pandas as pd
from pathlib import Path

In [10]:
evidence_path = Path('../reports/03_evidence/tp_evidence_packet.json')
manual_explanation_path = Path('../reports/04_llm_reasoning/manual_explanations/tp_manual_explanation.txt')

with open(evidence_path) as f:
    tp_evidence = json.load(f)

with open(manual_explanation_path) as f:
    tp_reference_explanation = f.read()

print('Evidence packet loaded.')
print('Reference explanation loaded.')

Evidence packet loaded.
Reference explanation loaded.


In [11]:
print(tp_reference_explanation)


1. Prediction summary
The model assigned this patient a very high predicted probability of in-hospital mortality (0.9995).

2. Main risk-increasing factors
The prediction was mainly driven upward by severe physiological instability. The strongest risk-increasing evidence was d1_heartrate_min = 0, which had the largest positive SHAP contribution. This value should be interpreted cautiously because a zero-valued heart rate may represent either an extreme clinical event or a recording artifact. Other major risk-increasing factors included very low minimum oxygen saturation (d1_spo2_min = 31), mechanical ventilation, very low systolic blood pressure (d1_sysbp_min = 41), low GCS motor score, low bicarbonate values, and low platelet count.

3. Main risk-decreasing factors
A few features slightly decreased the predicted risk, including d1_wbc_min, d1_bun_max, hematocrit-related features, BMI, and height. However, these negative contributions were much smaller than the strong risk-increasing 

In [12]:
evaluation_rubric = {
    'faithfulness_no_hallucination': {
        'description': 'The explanation should be grounded only in the provided evidence, should not invent unsupported facts, and should accurately reflect SHAP directions.',
        'score_1': 'Contains major unsupported claims, invents clinical facts, or contradicts the evidence.',
        'score_2': 'Uses some evidence but includes several unsupported, vague, or potentially hallucinated claims.',
        'score_3': 'Mostly grounded in the evidence, with minor unsupported phrasing or limited SHAP-direction precision.',
        'score_4': 'Clearly grounded in the evidence, does not invent major facts, and mostly reflects SHAP directions correctly.',
        'score_5': 'Fully faithful to the provided evidence, contains no unsupported claims, and accurately reflects SHAP directions and caution flags.'
    },
    'completeness': {
        'description': 'The explanation should cover the main risk-increasing evidence, risk-decreasing evidence, and overall prediction logic.',
        'score_1': 'Misses most key evidence.',
        'score_2': 'Covers only a few important factors.',
        'score_3': 'Covers main risk-increasing factors but gives limited attention to risk-decreasing evidence or overall interpretation.',
        'score_4': 'Covers most important risk-increasing and risk-decreasing factors with a clear overall interpretation.',
        'score_5': 'Comprehensively covers the main evidence, including risk-increasing factors, risk-decreasing factors, caution-relevant evidence, and overall interpretation.'
    },
    'clinical_plausibility': {
        'description': 'The explanation should interpret evidence in a clinically reasonable way.',
        'score_1': 'Clinically implausible or misleading interpretation.',
        'score_2': 'Some clinical reasoning is present but contains questionable interpretations.',
        'score_3': 'Generally plausible with some limited clinical depth.',
        'score_4': 'Clinically plausible and clearly interprets most important signals.',
        'score_5': 'Clinically strong, coherent, and appropriately cautious.'
    },
    'caution_awareness': {
        'description': 'The explanation should mention and correctly handle caution flags such as zero-valued vital signs or non-clinical identifiers.',
        'score_1': 'Ignores important caution flags.',
        'score_2': 'Mentions caution flags but handles them poorly.',
        'score_3': 'Mentions some caution flags but not all relevant ones.',
        'score_4': 'Correctly mentions and explains relevant caution flags.',
        'score_5': 'Fully integrates caution flags without overstating or ignoring them.'
    },
    'clarity': {
        'description': 'The explanation should be readable, structured, and easy to follow.',
        'score_1': 'Unclear or disorganized.',
        'score_2': 'Partly understandable but poorly structured.',
        'score_3': 'Understandable with adequate structure.',
        'score_4': 'Clear, organized, and easy to follow.',
        'score_5': 'Very clear, concise, and well structured.'
    }
}

In [13]:
rubric_df = pd.DataFrame(evaluation_rubric).T
rubric_df

,description,score_1,score_2,score_3,score_4,score_5
faithfulness_no_hallucination,The explanation should be grounded only in the...,"Contains major unsupported claims, invents cli...",Uses some evidence but includes several unsupp...,"Mostly grounded in the evidence, with minor un...","Clearly grounded in the evidence, does not inv...","Fully faithful to the provided evidence, conta..."
completeness,The explanation should cover the main risk-inc...,Misses most key evidence.,Covers only a few important factors.,Covers main risk-increasing factors but gives ...,Covers most important risk-increasing and risk...,"Comprehensively covers the main evidence, incl..."
clinical_plausibility,The explanation should interpret evidence in a...,Clinically implausible or misleading interpret...,Some clinical reasoning is present but contain...,Generally plausible with some limited clinical...,Clinically plausible and clearly interprets mo...,"Clinically strong, coherent, and appropriately..."
caution_awareness,The explanation should mention and correctly h...,Ignores important caution flags.,Mentions caution flags but handles them poorly.,Mentions some caution flags but not all releva...,Correctly mentions and explains relevant cauti...,Fully integrates caution flags without oversta...
clarity,"The explanation should be readable, structured...",Unclear or disorganized.,Partly understandable but poorly structured.,Understandable with adequate structure.,"Clear, organized, and easy to follow.","Very clear, concise, and well structured."


In [14]:
evaluation_weights = {
    'faithfulness_no_hallucination': 0.30,
    'clinical_plausibility': 0.25,
    'caution_awareness': 0.20,
    'completeness': 0.15,
    'clarity': 0.10
}

sum(evaluation_weights.values())

1.0

In [15]:
def compute_weighted_score(score_dict, weights):
    weighted_score = 0

    for metric, weight in weights.items():
        weighted_score += score_dict[metric]['score'] * weight

    return weighted_score

In [16]:
tp_reference_scores = {
    'faithfulness_no_hallucination': {
        'score': 5,
        'rationale': 'The explanation uses only the provided evidence, accurately reflects the SHAP direction of the main features, and does not introduce unsupported clinical facts.'
    },
    'completeness': {
        'score': 5,
        'rationale': 'The explanation covers the main risk-increasing factors, mentions risk-decreasing factors, includes caution notes, and provides an overall interpretation.'
    },
    'clinical_plausibility': {
        'score': 5,
        'rationale': 'The clinical interpretation is coherent and plausible, linking hypoxemia, mechanical ventilation, hypotension, impaired GCS, low bicarbonate, and thrombocytopenia to increased mortality risk.'
    },
    'caution_awareness': {
        'score': 5,
        'rationale': 'The explanation explicitly identifies d1_heartrate_min = 0 as a caution flag and avoids overinterpreting it.'
    },
    'clarity': {
        'score': 5,
        'rationale': 'The explanation is structured, concise, and easy to follow.'
    }
}

In [17]:
tp_reference_eval_df = pd.DataFrame(tp_reference_scores).T
tp_reference_eval_df

,score,rationale
faithfulness_no_hallucination,5,The explanation uses only the provided evidenc...
completeness,5,The explanation covers the main risk-increasin...
clinical_plausibility,5,The clinical interpretation is coherent and pl...
caution_awareness,5,The explanation explicitly identifies d1_heart...
clarity,5,"The explanation is structured, concise, and ea..."


In [18]:
tp_reference_overall_score = compute_weighted_score(
    tp_reference_scores,
    evaluation_weights
)

tp_reference_overall_score

5.0

In [19]:
os.makedirs('../reports/05_evaluation', exist_ok=True)

rubric_df.to_csv('../reports/05_evaluation/explanation_evaluation_rubric.csv')
tp_reference_eval_df.to_csv('../reports/05_evaluation/tp_reference_evaluation.csv')

with open('../reports/05_evaluation/tp_reference_explanation.txt', 'w') as f:
    f.write(tp_reference_explanation)

with open('../reports/05_evaluation/evaluation_weights.json', 'w') as f:
    json.dump(evaluation_weights, f, indent=2)

with open('../reports/05_evaluation/tp_reference_overall_score.json', 'w') as f:
    json.dump({'overall_score': tp_reference_overall_score}, f, indent=2)

print('Evaluation framework saved.')

Evaluation framework saved.
